## Module imports

In [1]:
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.transform import rowcol
from pyproj import Transformer
from pathlib import Path
import geopandas as gpd
import pandas as pd
from scipy.ndimage import binary_dilation, distance_transform_edt

In [2]:
np.random.seed(42)

# Basic Data Exploration for geographical alignment

In [3]:
# defining paths
path_base_sentinel = Path.cwd() / "data" / "cleaned data" 
path_base_baumk = Path.cwd() / "data" / "cleaned data" 
path_base_prep = Path.cwd() / "data" / "preprocessed data"

In [4]:
def explore_images(paths):
    # function to explore if the images of all months are geographically aligned
    for path in paths:
        with rasterio.open(path) as src:
            print("bounds:", src.bounds)
            print("res:", src.res)
            print("transform:", src.transform)
            print("shape:", (src.width, src.height))
            print("crs:", (src.crs))
            print("----")

## Barcelona Data Exploration

In [5]:
path_bar_apr = path_base_sentinel / "S2_barcelona_apr_25.tif"
path_bar_aug = path_base_sentinel / "S2_barcelona_aug_25.tif"
path_bar_nov = path_base_sentinel / "S2_barcelona_nov_25.tif"
paths_bar = [path_bar_apr,path_bar_aug,path_bar_nov]

In [6]:
# check if the images for all months are perfectly geographically aligned
explore_images(paths_bar)

bounds: BoundingBox(left=2.09100848684501, bottom=41.344960951600974, right=2.2251269587640548, top=41.46785048246853)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 2.09|
| 0.00,-0.00, 41.47|
| 0.00, 0.00, 1.00|
shape: (1493, 1368)
crs: EPSG:4326
----
bounds: BoundingBox(left=2.09100848684501, bottom=41.344960951600974, right=2.2251269587640548, top=41.46785048246853)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 2.09|
| 0.00,-0.00, 41.47|
| 0.00, 0.00, 1.00|
shape: (1493, 1368)
crs: EPSG:4326
----
bounds: BoundingBox(left=2.09100848684501, bottom=41.344960951600974, right=2.2251269587640548, top=41.46785048246853)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 2.09|
| 0.00,-0.00, 41.47|
| 0.00, 0.00, 1.00|
shape: (1493, 1368)
crs: EPSG:4326
----


In [7]:
# check if tree data has the same coordinate system, this is not a geoparquet file therefore manual checking of head necessary
path_bar_tree = path_base_baumk / "barcelona.parquet"
gdf_bar = pd.read_parquet(path_bar_tree)
gdf_bar.head()

,latitude,longitude,species_latin,type_of_tree,city
0,41.436508,2.170511,Phoenix dactylifera,PALMERA ZONA,barcelona
1,41.437769,2.162530,Platanus × acerifolia,ARBRE ZONA,barcelona
2,41.437788,2.162501,Platanus × acerifolia,ARBRE ZONA,barcelona
3,41.437810,2.162471,Platanus × acerifolia,ARBRE ZONA,barcelona
4,41.435251,2.169097,Pinus halepensis,ARBRE ZONA,barcelona


## Hamburg Data Exploration

In [8]:
path_ham_apr = path_base_sentinel / "S2_hamburg_apr_25.tif"
path_ham_aug = path_base_sentinel / "S2_hamburg_aug_25.tif"
path_ham_nov = path_base_sentinel / "S2_hamburg_nov_25.tif"
paths_ham = [path_ham_apr,path_ham_aug,path_ham_nov]

In [9]:
# check if the images for all months are perfectly geographically aligned
explore_images(paths_ham)

bounds: BoundingBox(left=9.854788161376387, bottom=53.462515482146024, right=10.0698448403946, top=53.54417234147249)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 9.85|
| 0.00,-0.00, 53.54|
| 0.00, 0.00, 1.00|
shape: (2394, 909)
crs: EPSG:4326
----
bounds: BoundingBox(left=9.854788161376387, bottom=53.462515482146024, right=10.0698448403946, top=53.54417234147249)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 9.85|
| 0.00,-0.00, 53.54|
| 0.00, 0.00, 1.00|
shape: (2394, 909)
crs: EPSG:4326
----
bounds: BoundingBox(left=9.854788161376387, bottom=53.462515482146024, right=10.0698448403946, top=53.54417234147249)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 9.85|
| 0.00,-0.00, 53.54|
| 0.00, 0.00, 1.00|
shape: (2394, 909)
crs: EPSG:4326
----


In [10]:
# check if tree data has the same coordinate system
path_ham_tree = path_base_baumk / "hamburg.parquet"
gdf_ham = gpd.read_parquet(path_ham_tree)
print(gdf_ham.crs)

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal

## Innsbruck Data Exploration

In [11]:
path_inn_apr = path_base_sentinel / "S2_innsbruck_apr_25.tif"
path_inn_aug = path_base_sentinel / "S2_innsbruck_aug_25.tif"
path_inn_nov = path_base_sentinel / "S2_innsbruck_nov_25.tif"
paths_inn = [path_inn_apr,path_inn_aug,path_inn_nov]

In [12]:
# check if the images for all months are perfectly geographically aligned
explore_images(paths_inn)

bounds: BoundingBox(left=11.324791292309571, bottom=47.22479381227689, right=11.451992736540896, top=47.28956234426191)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 11.32|
| 0.00,-0.00, 47.29|
| 0.00, 0.00, 1.00|
shape: (1416, 721)
crs: EPSG:4326
----
bounds: BoundingBox(left=11.324791292309571, bottom=47.22479381227689, right=11.451992736540896, top=47.28956234426191)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 11.32|
| 0.00,-0.00, 47.29|
| 0.00, 0.00, 1.00|
shape: (1416, 721)
crs: EPSG:4326
----
bounds: BoundingBox(left=11.324791292309571, bottom=47.22479381227689, right=11.451992736540896, top=47.28956234426191)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 11.32|
| 0.00,-0.00, 47.29|
| 0.00, 0.00, 1.00|
shape: (1416, 721)
crs: EPSG:4326
----


In [13]:
# set coordinate system to the same as sentinel-2
path_inn_tree = path_base_baumk / "innsbruck.parquet"
df_inn = pd.read_parquet(path_inn_tree)
gdf_inn = gpd.GeoDataFrame(df_inn,geometry=gpd.points_from_xy(df_inn.longitude, df_inn.latitude),crs="EPSG:4326")


## Linz Data Exploration

In [14]:
path_lin_apr = path_base_sentinel / "S2_linz_apr_25.tif"
path_lin_aug = path_base_sentinel / "S2_linz_aug_25.tif"
path_lin_okt = path_base_sentinel / "S2_linz_okt_25.tif"
paths_lin = [path_lin_apr,path_lin_aug,path_lin_okt]

In [15]:
# check if the images for all months are perfectly geographically aligned
explore_images(paths_lin)

bounds: BoundingBox(left=14.247549900720847, bottom=48.21608472830278, right=14.39128034617997, top=48.35262865148895)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 14.25|
| 0.00,-0.00, 48.35|
| 0.00, 0.00, 1.00|
shape: (1600, 1520)
crs: EPSG:4326
----
bounds: BoundingBox(left=14.247549900720847, bottom=48.21608472830278, right=14.39128034617997, top=48.35262865148895)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 14.25|
| 0.00,-0.00, 48.35|
| 0.00, 0.00, 1.00|
shape: (1600, 1520)
crs: EPSG:4326
----
bounds: BoundingBox(left=14.247549900720847, bottom=48.21608472830278, right=14.39128034617997, top=48.35262865148895)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 14.25|
| 0.00,-0.00, 48.35|
| 0.00, 0.00, 1.00|
shape: (1600, 1520)
crs: EPSG:4326
----


In [16]:
# set coordinate system to the same as sentinel-2
path_lin_tree = path_base_baumk / "linz.parquet"
df_lin = pd.read_parquet(path_lin_tree)
gdf_lin = gpd.GeoDataFrame(df_lin,geometry=gpd.points_from_xy(df_lin.longitude, df_lin.latitude),crs="EPSG:4326")

## Paris Data Exploration

In [17]:
path_par_apr = path_base_sentinel / "S2_paris_apr_25.tif"
path_par_aug = path_base_sentinel / "S2_paris_aug_25.tif"
path_par_okt = path_base_sentinel / "S2_paris_okt_25.tif"
paths_par = [path_par_apr,path_par_aug,path_par_okt]

In [18]:
# check if the images for all months are perfectly geographically aligned
explore_images(paths_par)

bounds: BoundingBox(left=2.2102149250476706, bottom=48.74222799021159, right=2.469828042158212, top=48.91227907349541)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 2.21|
| 0.00,-0.00, 48.91|
| 0.00, 0.00, 1.00|
shape: (2890, 1893)
crs: EPSG:4326
----
bounds: BoundingBox(left=2.2102149250476706, bottom=48.74222799021159, right=2.469828042158212, top=48.91227907349541)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 2.21|
| 0.00,-0.00, 48.91|
| 0.00, 0.00, 1.00|
shape: (2890, 1893)
crs: EPSG:4326
----
bounds: BoundingBox(left=2.2102149250476706, bottom=48.74222799021159, right=2.469828042158212, top=48.91227907349541)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 2.21|
| 0.00,-0.00, 48.91|
| 0.00, 0.00, 1.00|
shape: (2890, 1893)
crs: EPSG:4326
----


In [19]:
# check if tree data has the same coordinate system
path_par_tree = path_base_baumk / "paris.parquet"
gdf_par = gpd.read_parquet(path_par_tree)
print(gdf_par.crs)

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal

## Vienna Data Exploration

In [20]:
path_vie_apr = path_base_sentinel / "S2_vienna_apr_25.tif"
path_vie_aug = path_base_sentinel / "S2_vienna_aug_25.tif"
path_vie_nov = path_base_sentinel / "S2_vienna_nov_25.tif"
paths_vie = [path_vie_apr,path_vie_aug,path_vie_nov]

In [21]:
# check if the images for all months are perfectly geographically aligned
explore_images(paths_vie)

bounds: BoundingBox(left=16.123950866189702, bottom=48.12283960181118, right=16.569605078641395, top=48.32001980667541)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 16.12|
| 0.00,-0.00, 48.32|
| 0.00, 0.00, 1.00|
shape: (4961, 2195)
crs: EPSG:4326
----
bounds: BoundingBox(left=16.123950866189702, bottom=48.12283960181118, right=16.569605078641395, top=48.32001980667541)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 16.12|
| 0.00,-0.00, 48.32|
| 0.00, 0.00, 1.00|
shape: (4961, 2195)
crs: EPSG:4326
----
bounds: BoundingBox(left=16.123950866189702, bottom=48.12283960181118, right=16.569605078641395, top=48.32001980667541)
res: (8.983152841195215e-05, 8.983152841195215e-05)
transform: | 0.00, 0.00, 16.12|
| 0.00,-0.00, 48.32|
| 0.00, 0.00, 1.00|
shape: (4961, 2195)
crs: EPSG:4326
----


In [22]:
# check if tree data has the same coordinate system
path_vie_tree = path_base_baumk / "vienna.parquet"
gdf_vie = gpd.read_parquet(path_vie_tree)
print(gdf_vie.crs)

{"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "degree"}]}, "scope": "Horizontal

# Data Processing

In [23]:
def align_to_reference(src_path, ref_profile):
    # this function loads the images of different months and aligns them geographically with oneanother

    # read in data and fix nan values
    with rasterio.open(src_path) as src:
        src_data = src.read(masked=True).astype(np.float32)
        src_data = src_data.filled(np.nan)
        src_data = np.nan_to_num(src_data, nan=0.0)

        # reorder bands and drop unnecessary, only keep R,G,B,NIR,NDVI
        src_data = src_data[[2, 1, 0, 3, 4]]

        # this part is to get all months on the same grid
        dst = np.zeros(
            (src_data.shape[0], ref_profile["height"], ref_profile["width"]),
            dtype=np.float32
        )

        for c in range(src_data.shape[0]):
            reproject(
                source=src_data[c],
                destination=dst[c],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_profile["transform"],
                dst_crs=ref_profile["crs"],
                resampling=Resampling.nearest
            )

    return dst

In [24]:
def load_time_series_aligned(paths):
    # load the first month as a reference
    with rasterio.open(paths[0]) as ref:
        ref_profile = {
            "transform": ref.transform,
            "crs": ref.crs,
            "width": ref.width,
            "height": ref.height
        }

    # create a list of all months
    imgs = []
    for p in paths:
        imgs.append(align_to_reference(p, ref_profile))

    return np.stack(imgs, axis=0), ref_profile

In [25]:
def normalize_per_band_time(img):
    # this function normalizes the image values for better ML-performance
    
    norm_img = np.zeros_like(img, dtype=np.float32)
    T, C, H, W = img.shape

    for c in range(C):
        band = img[:, c, :, :]

        min_val = np.percentile(band, 2)
        max_val = np.percentile(band, 98)

        norm_img[:, c, :, :] = np.clip((band - min_val) / (max_val - min_val + 1e-6),0, 1)

    return norm_img

In [26]:
def convert_tree_coords(tree_coords, ref_profile):
    # convert the coordinates of trees into pixel values using the reference image created in load_time_series_aligned
    
    transformer = Transformer.from_crs("EPSG:4326",ref_profile["crs"],always_xy=True)

    tree_pixels = []

    for lon, lat in tree_coords:
        x, y = transformer.transform(lon, lat)
        r, c = rowcol(ref_profile["transform"], x, y)
        tree_pixels.append((r, c))

    return np.array(tree_pixels)

In [28]:
def extract_patches_time_with_tree_counts(img,tree_pixels,patch_size=256,stride=128,distance=5):
    # patch each image and get all tree locations in those new images

    T, C, H, W = img.shape

    patches = []
    masks = []
    proximity_masks = []

    tree_pixels = np.array(tree_pixels)

    # circular structuring element to find 5 pixel radius
    yy, xx = np.ogrid[-distance:distance+1, -distance:distance+1]
    structure = (xx**2 + yy**2) <= distance**2

    for i in range(0, H - patch_size + 1, stride):
        for j in range(0, W - patch_size + 1, stride):

            patch = img[:, :, i:i+patch_size, j:j+patch_size]

            mask = np.zeros((patch_size, patch_size), dtype=np.float32)

            # fast filtering (vectorized)
            inside = (
                (tree_pixels[:, 0] >= i) &
                (tree_pixels[:, 0] < i + patch_size) &
                (tree_pixels[:, 1] >= j) &
                (tree_pixels[:, 1] < j + patch_size)
            )

            selected = tree_pixels[inside]

            # convert to local patch coords
            for r, c in selected:
                lr = r - i
                lc = c - j

                # COUNT trees per pixel (important change!)
                mask[lr, lc] += 1
            
            # binary tree presence
            binary_mask = mask > 0

            # create distance-based target mask
            proximity_mask = binary_dilation(binary_mask, structure=structure).astype(np.float32)

            patches.append(patch)
            masks.append(mask)
            proximity_masks.append(proximity_mask)

    return np.array(patches), np.array(masks), np.array(proximity_masks)

In [29]:
def delete_sparse_patches(patches_img, patches_tree, patches_true, min_trees=0):
    tree_counts = patches_tree.sum(axis=(1, 2))
    keep_mask = tree_counts > min_trees
    return (patches_img[keep_mask], patches_tree[keep_mask], patches_true[keep_mask])

In [30]:
def preprocess(paths, tree_coords, patch_size=256, stride=128, augment = True):
    # 1. load + align
    img, ref_profile = load_time_series_aligned(paths)

    # 2. normalize
    img = normalize_per_band_time(img)

    # 3. convert trees to pixel space
    tree_pixels = convert_tree_coords(tree_coords, ref_profile)

    # 4. patch + labels
    patches_img, patches_tree, patches_usable = extract_patches_time_with_tree_counts(img,tree_pixels,patch_size,stride)

    # 5. delete all patches with only zero labels
    delete_sparse_patches(patches_img, patches_tree, patches_usable, min_trees=15)

    return patches_img, patches_tree, patches_usable

## Barcelona Data Processing

In [31]:
# get tree coords
tree_coords_bar = gdf_bar[["longitude", "latitude"]].to_numpy()

# get the patched sentinel images and tree coords
imgs_bar, trees_bar, usables_bar = preprocess(paths_bar,tree_coords_bar)

# create relative positions in the year according to the used months
dates_bar = np.tile([3/12, 7/12, 10/12], (imgs_bar.shape[0], 1))

In [32]:
# save the data
np.save(path_base_prep / "satellite_patches_bar.npy", imgs_bar)
np.save(path_base_prep / "tree_patches_bar.npy",trees_bar)
np.save(path_base_prep / "date_patches_bar.npy",dates_bar)
np.save(path_base_prep / "usable_patches_bar.npy",usables_bar)

## Hamburg Data Processing

In [33]:
# get tree coords
tree_coords_ham = gdf_ham[["longitude", "latitude"]].to_numpy()

# get the patched sentinel images and tree coords
imgs_ham, trees_ham, usables_ham = preprocess(paths_ham,tree_coords_ham)

# create relative positions in the year according to the used months
dates_ham = np.tile([3/12, 7/12, 10/12], (imgs_ham.shape[0], 1))

In [34]:
# save the data
np.save(path_base_prep / "satellite_patches_ham.npy", imgs_ham)
np.save(path_base_prep / "tree_patches_ham.npy",trees_ham)
np.save(path_base_prep / "date_patches_ham.npy",dates_ham)
np.save(path_base_prep / "usable_patches_ham.npy",usables_ham)

## Innsbruck Data Processing

In [35]:
# get tree coords
tree_coords_inn = gdf_inn[["longitude", "latitude"]].to_numpy()

# get the patched sentinel images and tree coords
imgs_inn, trees_inn, usables_inn = preprocess(paths_inn,tree_coords_inn)

# create relative positions in the year according to the used months
dates_inn = np.tile([3/12, 7/12, 10/12], (imgs_inn.shape[0], 1))

In [36]:
# save the data
np.save(path_base_prep / "satellite_patches_inn.npy", imgs_inn)
np.save(path_base_prep / "tree_patches_inn.npy",trees_inn)
np.save(path_base_prep / "date_patches_inn.npy",dates_inn)
np.save(path_base_prep / "usable_patches_inn.npy",usables_inn)

## Linz Data Processing

In [37]:
# get tree coords
tree_coords_lin = gdf_lin[["longitude", "latitude"]].to_numpy()

# get the patched sentinel images and tree coords
imgs_lin, trees_lin, usables_lin = preprocess(paths_lin,tree_coords_lin)

# create relative positions in the year according to the used months
dates_lin = np.tile([3/12, 7/12, 9/12], (imgs_lin.shape[0], 1))

In [38]:
# save the data
np.save(path_base_prep / "satellite_patches_lin.npy", imgs_lin)
np.save(path_base_prep / "tree_patches_lin.npy",trees_lin)
np.save(path_base_prep / "date_patches_lin.npy",dates_lin)
np.save(path_base_prep / "usable_patches_lin.npy",usables_lin)

## Paris Data Processing

In [39]:
# get tree coords
tree_coords_par = gdf_par[["longitude", "latitude"]].to_numpy()

# get the patched sentinel images and tree coords
imgs_par, trees_par, usables_par = preprocess(paths_par,tree_coords_par)

# create relative positions in the year according to the used months
dates_par = np.tile([3/12, 7/12, 9/12], (imgs_par.shape[0], 1))

In [40]:
# save the data
np.save(path_base_prep / "satellite_patches_par.npy", imgs_par)
np.save(path_base_prep / "tree_patches_par.npy",trees_par)
np.save(path_base_prep / "date_patches_par.npy",dates_par)
np.save(path_base_prep / "usable_patches_par.npy",usables_par)

## Vienna Data Processing

In [41]:
# get tree coords
tree_coords_vie = gdf_vie[["longitude", "latitude"]].to_numpy()

# get the patched sentinel images and tree coords
imgs_vie, trees_vie, usables_vie = preprocess(paths_vie,tree_coords_vie)

# create relative positions in the year according to the used months
dates_vie = np.tile([3/12, 7/12, 10/12], (imgs_vie.shape[0], 1))

In [42]:
# save the data
np.save(path_base_prep / "satellite_patches_vie.npy", imgs_vie)
np.save(path_base_prep / "tree_patches_vie.npy",trees_vie)
np.save(path_base_prep / "date_patches_vie.npy",dates_vie)
np.save(path_base_prep / "usable_patches_vie.npy",usables_vie)